# 07 — Phoenix H3 Feature Panel

Visualises the per-H3-cell feature time series built by `ug features build`.

**Sections**
1. Built-pct change map 2018 → 2025  
2. Top-50 fastest-growing cells  
3. Key feature distributions  
4. Google Earth sanity check — export centroids as KML

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from dotenv import load_dotenv

load_dotenv()

from urbangrowth.config import data_path, get_pipeline
from urbangrowth.processing.h3_aggregate import _latlng_to_cell, _cell_to_latlng, get_city_h3_cells

try:
    import h3
    import h3.api.numpy_int as h3int
except ImportError:
    pass

CITY       = 'phoenix'
START_DATE = '2018-01'
END_DATE   = '2025-12'
RESOLUTION = 8

pipe        = get_pipeline()
feat_dir    = data_path(pipe['processed_data_subdirs']['h3_features'], CITY)
panel_path  = data_path('processed', 'features') / f'{CITY}_h3_features.parquet'

print(f'Feature dir  : {feat_dir}')
print(f'Panel path   : {panel_path}')
print(f'Panel exists : {panel_path.exists()}')

lc_parquets = sorted(feat_dir.glob('*_lc.parquet'))
print(f'LC parquets  : {len(lc_parquets)}')

## 1. Built-pct Change Map 2018 → 2025

Expected hotspots:
- **TSMC fab corridor** — north Phoenix / Scottsdale (around 33.68°N, 111.97°W)
- **Buckeye** — far western suburbs (~33.37°N, 112.58°W)
- **Queen Creek / SE Mesa** — southeastern growth edge (~33.25°N, 111.63°W)

In [ ]:
def _load_lc_parquet(path: Path) -> pd.DataFrame:
    """Load a *_lc.parquet and return h3_index + built_pct."""
    df = pd.read_parquet(path, columns=['h3_index', 'built_pct'])
    return df


def _cell_centroid(cell: str):
    """Return (lat, lon) centroid of an H3 cell."""
    try:
        return _cell_to_latlng(cell)
    except Exception:
        return (np.nan, np.nan)


# Load first and last available month in LC parquets
def _month_label(p: Path) -> str:
    return p.name[:7]


early_parquets = [p for p in lc_parquets if _month_label(p) <= '2018-06']
late_parquets  = [p for p in lc_parquets if _month_label(p) >= '2025-06']

# Fall back to first/last if the date range doesn't span as expected
if not early_parquets and lc_parquets:
    early_parquets = lc_parquets[:1]
if not late_parquets and lc_parquets:
    late_parquets = lc_parquets[-1:]

if not early_parquets or not late_parquets:
    print('No LC parquets found. Run: ug process h3 --city phoenix')
else:
    early_label = _month_label(early_parquets[0])
    late_label  = _month_label(late_parquets[-1])

    # Average over the first few months for stability
    early_df = pd.concat([_load_lc_parquet(p) for p in early_parquets[:3]])
    early_df = early_df.groupby('h3_index', as_index=False)['built_pct'].mean()

    late_df  = pd.concat([_load_lc_parquet(p) for p in late_parquets[-3:]])
    late_df  = late_df.groupby('h3_index', as_index=False)['built_pct'].mean()

    merged = pd.merge(
        early_df.rename(columns={'built_pct': 'built_early'}),
        late_df.rename(columns={'built_pct': 'built_late'}),
        on='h3_index',
    )
    merged['delta_built'] = merged['built_late'] - merged['built_early']

    # Centroids
    centroids = [_cell_centroid(c) for c in merged['h3_index']]
    merged['lat'] = [c[0] for c in centroids]
    merged['lon'] = [c[1] for c in centroids]
    merged = merged.dropna(subset=['lat', 'lon'])

    print(f'Cells with data: {len(merged):,}')
    print(f'Period: {early_label} → {late_label}')
    print(merged['delta_built'].describe().round(4))

    # ── Map ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    vmax_abs = merged['delta_built'].abs().quantile(0.98)
    norm = mcolors.TwoSlopeNorm(vmin=-vmax_abs, vcenter=0, vmax=vmax_abs)

    for ax, (col, title, cmap) in zip(
        axes,
        [
            ('built_late',  f'Built fraction {late_label}',      'YlOrRd'),
            ('delta_built', f'Built Δ  {early_label} → {late_label}', 'RdYlGn_r'),
        ],
    ):
        if col == 'delta_built':
            sc = ax.scatter(
                merged['lon'], merged['lat'],
                c=merged[col], cmap=cmap, norm=norm,
                s=4, linewidths=0, alpha=0.85,
            )
        else:
            sc = ax.scatter(
                merged['lon'], merged['lat'],
                c=merged[col], cmap=cmap,
                vmin=0, vmax=merged[col].quantile(0.98),
                s=4, linewidths=0, alpha=0.85,
            )
        plt.colorbar(sc, ax=ax, fraction=0.04, label=col)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_aspect('equal')

        # Annotate known hotspots
        hotspots = {
            'TSMC':       (33.683, -111.970),
            'Buckeye':    (33.370, -112.583),
            'Queen Creek':(33.248, -111.634),
        }
        for name, (plat, plon) in hotspots.items():
            ax.annotate(
                name,
                xy=(plon, plat), xytext=(plon + 0.05, plat + 0.05),
                fontsize=7, color='navy',
                arrowprops=dict(arrowstyle='->', color='navy', lw=0.8),
            )

    fig.suptitle('Phoenix — Urban Expansion 2018 → 2025', fontsize=13)
    plt.tight_layout()
    plt.show()

## 2. Top-50 Fastest-Growing Cells

In [ ]:
if 'merged' in dir() and not merged.empty:
    top50 = (
        merged
        .nlargest(50, 'delta_built')
        [['h3_index', 'lat', 'lon', 'built_early', 'built_late', 'delta_built']]
        .reset_index(drop=True)
    )
    top50.index += 1
    top50.columns = ['H3 Index', 'Lat', 'Lon', 'Built % (early)', 'Built % (late)', 'Δ Built pp']
    top50[['Built % (early)', 'Built % (late)', 'Δ Built pp']] *= 100
    top50 = top50.round({'Lat': 4, 'Lon': 4, 'Built % (early)': 1, 'Built % (late)': 1, 'Δ Built pp': 1})

    print(f'Top-50 fastest-growing H3 cells ({early_label} → {late_label})')
    print(top50.to_string())

    # Scatter the top-50 on the delta map
    fig, ax = plt.subplots(figsize=(12, 9))
    sc = ax.scatter(
        merged['lon'], merged['lat'],
        c=merged['delta_built'],
        cmap='RdYlGn_r',
        norm=mcolors.TwoSlopeNorm(
            vmin=merged['delta_built'].quantile(0.02),
            vcenter=0,
            vmax=merged['delta_built'].quantile(0.98),
        ),
        s=4, linewidths=0, alpha=0.7,
    )
    plt.colorbar(sc, ax=ax, fraction=0.03, label='Δ built fraction')

    # Highlight top-50
    ax.scatter(
        top50['Lon'], top50['Lat'],
        s=40, facecolors='none', edgecolors='blue', linewidths=0.8,
        label='Top-50 growth cells',
    )
    for _, row in top50.head(10).iterrows():
        ax.annotate(
            str(row.name),
            xy=(row['Lon'], row['Lat']),
            xytext=(row['Lon'] + 0.02, row['Lat'] + 0.02),
            fontsize=6, color='blue',
        )

    ax.set_title(f'Phoenix — Top-50 Growth Cells Highlighted  ({early_label} → {late_label})',
                 fontsize=11)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Run the cell above first.')

## 3. Key Feature Distributions

Load the assembled panel (if it exists) and plot distributions of the main features.

In [ ]:
if panel_path.exists():
    panel = pd.read_parquet(panel_path)
    print(f'Panel shape: {panel.shape}')
    print(f'Columns    : {list(panel.columns)}')
    print(f'Date range : {panel["date"].min()} → {panel["date"].max()}')
    print(f'H3 cells   : {panel["h3_index"].nunique():,}')
    print()

    # Time series of mean built_pct across all cells
    if 'built_pct' in panel.columns:
        monthly_built = panel.groupby('date')['built_pct'].mean()

        fig, ax = plt.subplots(figsize=(16, 3.5))
        ax.plot(range(len(monthly_built)), monthly_built.values * 100,
                marker='o', ms=3, linewidth=1.5, color='#C44E52')
        tick_pos = list(range(0, len(monthly_built), max(1, len(monthly_built) // 16)))
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([str(monthly_built.index[i]) for i in tick_pos],
                           rotation=45, ha='right')
        ax.set_ylabel('Mean built fraction (%)')
        ax.set_title('Phoenix — mean H3-cell built fraction over time')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Feature correlation heatmap (most recent month)
    numeric_cols = [
        c for c in panel.columns
        if panel[c].dtype in ('float32', 'float64', 'int32', 'int64')
        and c not in ('pixel_count',)
    ]
    if numeric_cols:
        latest = panel[panel['date'] == panel['date'].max()][numeric_cols].dropna(axis=1, how='all')
        if len(latest.columns) > 1:
            corr = latest.corr()
            fig, ax = plt.subplots(figsize=(min(18, len(corr) * 0.8), min(15, len(corr) * 0.65)))
            im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
            ax.set_xticks(range(len(corr)))
            ax.set_yticks(range(len(corr)))
            ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=7)
            ax.set_yticklabels(corr.columns, fontsize=7)
            plt.colorbar(im, ax=ax, fraction=0.03)
            ax.set_title(f'Feature correlation — {panel["date"].max()}', fontsize=11)
            plt.tight_layout()
            plt.show()
else:
    print(f'Panel not found at {panel_path}')
    print('Run: ug features build --city phoenix')
    print()

    # Fallback: load monthly LC parquets directly
    if lc_parquets:
        all_built = []
        for p in lc_parquets:
            df = pd.read_parquet(p, columns=['h3_index', 'built_pct', 'date'])
            all_built.append(df['built_pct'].mean())
        dates = [_month_label(p) for p in lc_parquets]

        fig, ax = plt.subplots(figsize=(16, 3.5))
        ax.plot(range(len(all_built)), [v * 100 for v in all_built],
                marker='o', ms=3, linewidth=1.5, color='#C44E52')
        tick_pos = list(range(0, len(dates), max(1, len(dates) // 16)))
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([dates[i] for i in tick_pos], rotation=45, ha='right')
        ax.set_ylabel('Mean built fraction (%)')
        ax.set_title('Phoenix — mean H3-cell built fraction (from LC parquets)')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

## 4. Google Earth Sanity Check — Export Top-50 as KML

Exports a KML file of the top-50 growth cells so you can drag-and-drop into Google Earth
to visually confirm alignment with known construction corridors.

In [ ]:
if 'top50' in dir() and not top50.empty:
    kml_path = Path('outputs') / 'phoenix_top50_growth_cells.kml'
    kml_path.parent.mkdir(exist_ok=True)

    lines = [
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<kml xmlns="http://www.opengis.net/kml/2.2">',
        '<Document>',
        f'  <name>Phoenix Top-50 Growth Cells ({early_label} → {late_label})</name>',
        '  <Style id="top50">',
        '    <IconStyle><color>ff0000ff</color><scale>0.9</scale>',
        '      <Icon><href>http://maps.google.com/mapfiles/kml/paddle/red-circle.png</href></Icon>',
        '    </IconStyle>',
        '  </Style>',
    ]

    for rank, row in top50.iterrows():
        lines += [
            '  <Placemark>',
            f'    <name>#{rank} Δ={row["Δ Built pp"]:.1f}pp</name>',
            f'    <description>H3: {row["H3 Index"]}\nBuilt early: {row["Built % (early)"]:.1f}%\n'
            f'Built late: {row["Built % (late)"]:.1f}%\nΔ: {row["Δ Built pp"]:.1f}pp</description>',
            '    <styleUrl>#top50</styleUrl>',
            '    <Point>',
            f'      <coordinates>{row["Lon"]},{row["Lat"]},0</coordinates>',
            '    </Point>',
            '  </Placemark>',
        ]

    lines += ['</Document>', '</kml>']
    kml_path.write_text('\n'.join(lines), encoding='utf-8')
    print(f'KML saved → {kml_path.resolve()}')
    print('Open in Google Earth Pro or drag into maps.google.com')

    # Print lat/lon table for quick spot-check
    print()
    print('Top-10 cell centroids for manual verification:')
    print(top50.head(10)[['H3 Index', 'Lat', 'Lon', 'Δ Built pp']].to_string())
else:
    print('Run cells 3 and 5 first to compute the top-50.')